# Aviator: tratamento e geração de dados

1. Lê os payouts coletados (`data/payouts.txt`);
2. Ajusta uma distribuição de Pareto aos dados reais;
3. Gera ~45k valores sintéticos e exporta tudo em CSV e Excel.

In [32]:
import csv

import pandas as pd
from scipy.stats import pareto

from main import read_data, treat_data


In [33]:
# CONFIG: limite para clean_data (odds acima são outliers)
LIMIT = 50

In [34]:
file_path = "data/payouts.txt"
data = read_data(file_path)
treated_data = treat_data(data)


In [35]:
# Estimate the shape, location, and scale parameters
shape, location, scale = pareto.fit(treated_data)

# Generate numbers: 9x a amostra real -> ~50k valores no total
random_numbers = pareto.rvs(shape, location, scale, size=9 * len(treated_data))
random_numbers = [round(n, 2) for n in random_numbers]


In [36]:
# Escrita dos valores originais
f = open('data/payouts.csv', 'w', encoding='utf-8', newline='')

writer = csv.writer(f)
for index, value in enumerate(treated_data):
    writer.writerow([index, value])

f.close()

# Escrita dos valores originais + gerados
f = open('data/payouts50k.csv', 'w', encoding='utf-8', newline='')

writer = csv.writer(f)
for index, value in enumerate(treated_data + random_numbers):
    writer.writerow([index, value])

f.close()


In [37]:
# Create a dictionary from the lists
data = {'Index': list(range(len(treated_data))), 'Valor': treated_data}

# Create a DataFrame from the dictionary
df = pd.DataFrame(data)

# Write the DataFrame to an Excel file
df.to_excel('data/payouts.xlsx', index=False)

# ------------------------------------------

# Create a dictionary from the lists
all_data = treated_data + random_numbers
data = {'Index': list(range(len(all_data))), 'Valor': all_data}

# Create a DataFrame from the dictionary
df = pd.DataFrame(data)

# Write the DataFrame to an Excel file
df.to_excel('data/payouts50k.xlsx', index=False)